# TerraPulse AI · 01: build the fine-tuning data (Colab)
Open this notebook in VS Code with the **Colab extension** and pick a **T4 GPU** runtime. Also runs on any Jupyter server with an NVIDIA GPU (e.g. a college DGX). Nothing here runs on your laptop.

1. intent data: generated with exact labels (`training/gen_intent_data.py`)
2. real fact sheets: the live pipeline (SLM off) on generated questions (500 on a big GPU server, 240 on Colab) (`training/build_compose_data.py`)
3. answer drafts: an **open** teacher (Qwen3-14B on a ≥40 GB GPU such as a DGX, Qwen3-4B on a T4; batched) writes answers from each fact sheet; only drafts that pass
   the runtime **grounding verifier** are kept. No hosted/proprietary LLM is used anywhere.
4. **chat style**: answers use the friendly assistant prompt (`chat_messages`), plus verified "why?" follow-up answers (`explain_messages`) so the model learns to converse.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the project into this runtime: Colab or any Jupyter server (e.g. a college DGX).
# Option A: clone your GitHub repo. Option B (Colab only): upload GEOAI.zip.
REPO_URL = "https://github.com/aaron43210/GeoAI_Hackathon.git"   # set "" to upload GEOAI.zip instead
REPO_BRANCH = "round2-dgx"       # round-2 work lives on this branch; main stays as the submitted version
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
WORK = "/content/GEOAI" if IN_COLAB else os.path.expanduser("~/GeoAI_Hackathon")   # its own folder (change if home is small)
if REPO_URL and os.path.isdir(WORK) and not os.path.exists(f"{WORK}/.git"):
    raise SystemExit(f"{WORK} exists but is not a git clone of the repo: rename/remove it or change WORK")
# GPUs: on a shared multi-GPU server, use the ones with the most free memory (set CUDA_VISIBLE_DEVICES yourself to override).
NUM_GPUS = 2                           # two free GPUs let the teacher be Qwen3-32B (split across them)
def _free_gpus(min_free_mb=20000):
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=True).stdout
    except Exception:
        return []
    gpus = [tuple(int(x) for x in l.split(",")) for l in out.strip().splitlines() if l.strip()]
    return [i for i, free in sorted(gpus, key=lambda g: -g[1]) if free >= min_free_mb]
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")     # same GPU numbering as nvidia-smi
if "CUDA_VISIBLE_DEVICES" not in os.environ and not IN_COLAB:
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in _free_gpus()[:NUM_GPUS]) or "0"
print("GPU(s):", os.environ.get("CUDA_VISIBLE_DEVICES", "default"))
if REPO_URL and os.path.exists(f"{WORK}/.git"):      # existing clone: switch it to REPO_BRANCH at its latest commit
    subprocess.run(["git", "-C", WORK, "fetch", "--depth", "1", "origin",
                    f"+{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}"], check=True)
    subprocess.run(["git", "-C", WORK, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, WORK], check=True)
elif not os.path.exists(WORK):
    from google.colab import files
    up = files.upload()                         # choose GEOAI.zip (project folder zipped, without node_modules)
    subprocess.run(["unzip", "-q", next(iter(up)), "-d", os.path.dirname(WORK)], check=True)
os.chdir(WORK)
print("working in", os.getcwd())
%pip -q install -r backend/requirements-core.txt
sys.path.insert(0, f"{WORK}/backend")

In [ ]:
!{sys.executable} training/gen_intent_data.py --n 3000 --out training/data/intent.jsonl
!head -c 600 training/data/intent.jsonl

### Real fact sheets from live APIs (paced to respect free-API limits: ≈30–40 min for 240, ≈1–1.5 h for 500)

In [ ]:
# Keep the notebook output small (thousands of log lines can crash Jupyter): the app's per-request logs are
# silenced (LOG_LEVEL=WARNING), everything goes to training/data/factsheets.log, and the cell prints one short
# progress line every 10 sheets. Safe to stop and re-run: it continues from the sheets already saved.
import os, subprocess, sys
N = 240 if IN_COLAB else 500            # Colab sessions time out; on a server collect more (still polite to the free APIs)
OUT, LOG = "training/data/factsheets.jsonl", "training/data/factsheets.log"
os.makedirs("training/data", exist_ok=True)
have = sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0
todo = max(0, N - have)
print(f"{have} sheets already saved, collecting {todo} more")
skipped = 0
if todo:
    with open(LOG, "a") as log:
        p = subprocess.Popen([sys.executable, "-u", "training/build_compose_data.py", "--n", str(todo), "--pause", "2",
                              "--seed", str(11 + have), "--out", OUT],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env=dict(os.environ, LOG_LEVEL="WARNING"))
        for line in p.stdout:
            log.write(line)
            if line.startswith(("skip", "error")):
                skipped += 1
            elif line.startswith("[") and "/" in line:
                done = int(line[1:line.index("/")])
                if done % 10 == 0 or done == todo:
                    print(f"{have + done}/{N} saved, {skipped} skipped · {line.strip()[:110]}", flush=True)
        p.wait()
    print("exit code", p.returncode, "· full log:", LOG)
print(sum(1 for _ in open(OUT)), "fact sheets in", OUT)

### Open teacher drafts → keep only answers that pass the grounding verifier
The teacher is picked from the GPU size: Qwen3-14B in bf16 (batch 16) on a ≥40 GB GPU, Qwen3-4B in fp16 (batch 8) on a T4. Safe to interrupt and re-run: the next cell resumes from `training/data/compose.jsonl`. If CUDA runs out of memory, lower `B` in the next cell.

In [ ]:
%pip -q install -U "transformers>=4.51" accelerate
import importlib.util, subprocess, sys
if importlib.util.find_spec("torch") is None:          # Colab has torch; a fresh server env may not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
import json, gc, torch
# If an earlier teacher (e.g. the old Qwen3-8B cell) is still in GPU memory, free it first.
for _name in ("teacher", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
from app.slm.prompts import chat_messages, explain_messages
from app.evidence.grounding import verify

# Open teacher (Apache-2.0, same Qwen3 chat format as the student), chosen by free GPU memory:
#   >= 90 GB free in total (e.g. two DGX GPUs): Qwen3-32B in bf16, split across them, batch 16
#   >= 35 GB free (one A100/H100):              Qwen3-14B in bf16, batch 16
#   T4 (16 GB):                                 Qwen3-4B in fp16, batch 8  (Qwen3-8B in 4-bit was ~9 min/sheet on a T4)
FREE = sum(torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count()))
if FREE >= 90e9:
    TEACHER, B = "Qwen/Qwen3-32B", 16
elif FREE >= 35e9:
    TEACHER, B = "Qwen/Qwen3-14B", 16
else:
    TEACHER, B = "Qwen/Qwen3-4B", 8
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"teacher: {TEACHER} | batch {B} | {DTYPE} | {torch.cuda.device_count()} GPU(s), {FREE / 1e9:.0f} GB free")
tok = AutoTokenizer.from_pretrained(TEACHER, padding_side="left")
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=DTYPE, device_map="auto")  # spreads over the GPUs
teacher.eval()

def draft_batch(sheets, temperature):
    texts = [tok.apply_chat_template(chat_messages(s), tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False) for s in sheets]
    enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = teacher.generate(**enc, max_new_tokens=320, do_sample=True, temperature=temperature,
                               top_p=0.9, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

In [ ]:
import os
rows = [json.loads(l) for l in open("training/data/factsheets.jsonl")]
# Keep answers from an interrupted earlier run that are still in memory but not yet on disk.
_mem = list(globals().get("kept") or [])
_disk = [json.loads(l) for l in open("training/data/compose.jsonl")] if os.path.exists("training/data/compose.jsonl") else []
_seen = {k["messages"][1]["content"] for k in _disk}
_extra = [k for k in _mem if k["messages"][1]["content"] not in _seen]
if _extra:
    with open("training/data/compose.jsonl", "a") as fh:
        for k in _extra: fh.write(json.dumps(k, ensure_ascii=False) + "\n")
    print("saved", len(_extra), "answers from the interrupted run")
kept, stats = [], {"teacher": 0, "template": 0, "dropped": 0}
if os.path.exists("training/data/compose.jsonl"):           # resume after an interruption
    kept = [json.loads(l) for l in open("training/data/compose.jsonl")]
    for k in kept: stats[k["source"]] += 1
done = {k["messages"][1]["content"] for k in kept}
todo = [r for r in rows if r["fact_sheet"] not in done]
print(len(kept), "kept so far,", len(todo), "to go")

def ok(r, a):
    return verify(a, r["fact_sheet"], set(r["evidence_ids"]), r["known_names"], r["must_mention"],
                  r["recommended"], r.get("evidence_values")).status == "passed"

def save(r, src, ans):
    msgs = chat_messages(r["fact_sheet"]) + [{"role": "assistant", "content": ans}]
    item = {"task": "compose", "source": src, "messages": msgs, "teacher": TEACHER if src == "teacher" else None}
    kept.append(item); stats[src] += 1
    with open("training/data/compose.jsonl", "a") as fh: fh.write(json.dumps(item, ensure_ascii=False) + "\n")

for i in range(0, len(todo), B):
    batch = todo[i:i + B]
    retry = []
    for r, a in zip(batch, draft_batch([r["fact_sheet"] for r in batch], 0.5)):
        save(r, "teacher", a) if ok(r, a) else retry.append(r)
    if retry:
        for r, a in zip(retry, draft_batch([r["fact_sheet"] for r in retry], 0.8)):
            if ok(r, a): save(r, "teacher", a)
            elif stats["template"] < 0.25 * len(rows): save(r, "template", r["template_answer"])
            else: stats["dropped"] += 1
    print(f"{min(i + B, len(todo))}/{len(todo)}", stats)
print("final", stats, "kept", len(kept))

In [ ]:
# Follow-up conversation data: "why?"-style questions answered from the same fact sheet (kept only if verified).
FOLLOWUPS = ["why this place and time?", "why not earlier in the day?", "is the air okay for this?",
             "will it rain?", "how far is it and how long to get there?", "is it safe for kids?",
             "what should I be careful about?", "why is it not recommended?"]
import random
random.seed(1)
explain_kept = 0
def draft_explain_batch(pairs, temperature=0.4):
    texts = [tok.apply_chat_template(explain_messages(s, q), tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False) for s, q in pairs]
    enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = teacher.generate(**enc, max_new_tokens=220, do_sample=True, temperature=temperature, top_p=0.9,
                               pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

pairs = [(r, random.choice(FOLLOWUPS)) for r in rows for _ in range(2)]
for i in range(0, len(pairs), B):
    batch = pairs[i:i + B]
    for (r, q), a in zip(batch, draft_explain_batch([(r["fact_sheet"], q) for r, q in batch])):
        g = verify(a, r["fact_sheet"], set(r["evidence_ids"]), r["known_names"], None, True, r.get("evidence_values"))
        if g.status == "passed" and a.strip():
            item = {"task": "explain", "source": "teacher", "teacher": TEACHER,
                    "messages": explain_messages(r["fact_sheet"], q) + [{"role": "assistant", "content": a}]}
            with open("training/data/compose.jsonl", "a") as fh: fh.write(json.dumps(item, ensure_ascii=False) + "\n")
            explain_kept += 1
    print(f"{min(i + B, len(pairs))}/{len(pairs)} follow-ups, kept {explain_kept}")

In [ ]:
# Hugging Face login: uses the Colab secret HF_TOKEN if available, otherwise asks (input is hidden, never saved).
import os
from huggingface_hub import login
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    from getpass import getpass
    token = getpass("Hugging Face write token: ")
login(token)
os.environ["HF_TOKEN"] = token
HF_USER = "aaronr43210"

In [ ]:
# Optional: publish the datasets (open licence) so the model card can link them
from huggingface_hub import HfApi
api = HfApi(); repo = f"{HF_USER}/terrapulse-sft-data"
api.create_repo(repo, repo_type="dataset", exist_ok=True)
for f in ["intent.jsonl", "compose.jsonl", "factsheets.jsonl"]:
    api.upload_file(path_or_fileobj=f"training/data/{f}", path_in_repo=f, repo_id=repo, repo_type="dataset")
print("https://huggingface.co/datasets/" + repo)